# Transfer Learning for Computer Vision — Caltech-101 (Keras on PyTorch)

Same pipeline as `../TransferLearning_Caltech101.ipynb` and
`4-transfer_101.py`, but **Keras 3 runs on the PyTorch backend** and the
images are loaded with `torchvision` instead of `tf.data`. TensorFlow is not
used. See `conversion.md` for what changed and why.

A pretrained **MobileNetV2** is used as a feature extractor, a new
classification head is trained on Caltech-101 (102 classes = 101 objects +
background), then the top layers of the backbone are fine-tuned. The final
model is saved as `caltech101_model.h5` (target: ≥ 85% validation accuracy).

Cells marked **🔍 Curiosity corner** are optional visualisations. You can
skip them; the main track doesn't depend on them.

> **Runtime → Change runtime type → GPU** before running.

## 0. Setup

Colab already ships Keras 3, PyTorch and torchvision. Upgrade Keras to be
safe, and install `gdown` to download the dataset from Google Drive.

In [ ]:
!pip install -q -U keras gdown

Download and unzip the `caltech-101.zip` file using the provided Google Drive
link. The dataset is expected to be extracted into a folder named
`101_ObjectCategories`.

In [ ]:
import gdown
import os
import shutil

# Google Drive ID for caltech-101.zip
drive_file_id = '1ci0p73yJ-ckdirRtgCAdjf53y2uPSaJN'
output_zip_file = 'caltech-101.zip'

# Download the file
gdown.download(f'https://drive.google.com/uc?id={drive_file_id}', output_zip_file, quiet=False)

# Unzip the main file, which creates a directory named 'caltech-101'
!unzip -q {output_zip_file} -d .

# Untar 101_ObjectCategories.tar.gz inside 'caltech-101'
extracted_dir = 'caltech-101'
os.chdir(extracted_dir)
!tar -xzf 101_ObjectCategories.tar.gz
os.chdir('..')

# Move '101_ObjectCategories' to the working directory (DATA_DIR below)
shutil.move(os.path.join(extracted_dir, '101_ObjectCategories'), '.')

# Clean up the intermediate directory and zip file
shutil.rmtree(extracted_dir)
os.remove(output_zip_file)

if os.path.exists('101_ObjectCategories'):
    print("Successfully downloaded and extracted '101_ObjectCategories'.")
else:
    print("Error: '101_ObjectCategories' directory not found after processing.")

### Select the PyTorch backend, imports and configuration

`KERAS_BACKEND` must be set **before** Keras is imported for the first time.
If this cell prints a backend other than `torch`, use
**Runtime → Restart session** and run it again.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # must come before `import keras`

import keras
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2

print("Keras:", keras.__version__, "| backend:", keras.backend.backend())
print("PyTorch:", torch.__version__, "| CUDA GPU:", torch.cuda.is_available())
assert keras.backend.backend() == "torch", "Restart the session and rerun"

DATA_DIR = "101_ObjectCategories"
MODEL_PATH = "caltech101_model.h5"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 15
UNFREEZE_LAYERS = 30
WEIGHTS = "imagenet"
NUM_WORKERS = 2  # Colab has 2 CPU cores for image decoding

keras.utils.set_random_seed(42)  # seeds Python, NumPy and PyTorch

## 1. Load and prepare the datasets

`torchvision.datasets.ImageFolder` reads one class per sub-folder, the same
layout `image_dataset_from_directory` expects. A seeded random permutation
gives an 80/20 train/validation split. Labels are integers, which pairs with
`sparse_categorical_crossentropy` later.

The transforms (augmentation, preprocessing) are attached in section 2; here
we only fix the split and read the class names.

In [ ]:
full_ds = datasets.ImageFolder(DATA_DIR)  # no transform: returns PIL images
class_names = full_ds.classes
num_classes = len(class_names)  # 101 objects + background

perm = torch.randperm(len(full_ds),
                      generator=torch.Generator().manual_seed(42)).tolist()
n_val = int(0.2 * len(full_ds))
val_idx, train_idx = perm[:n_val], perm[n_val:]

print("Images:", len(full_ds), "| classes:", num_classes)
print("Training:", len(train_idx), "| validation:", len(val_idx))

> ### 🔍 Curiosity corner — what's in Caltech-101?
> A random sample of training images with their class names. Notice the
> different sizes and aspect ratios: everything is resized to 224×224 later.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, i in zip(axes.flat, rng.choice(train_idx, size=12, replace=False)):
    img, label = full_ds[int(i)]
    ax.imshow(img)
    ax.set_title(f"{class_names[label]}\n{img.size[0]}×{img.size[1]}",
                 fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Caltech-101 images")
plt.tight_layout()
plt.show()

## 2. Data augmentation + preprocessing

Augmentation is applied **only to the training set**, with
`torchvision.transforms.v2`:

- `RandomResizedCrop`: crops a random 80–100% area of the image and
  resizes it to 224×224 (the crop/zoom step, like Keras' `RandomZoom`)
- `RandomHorizontalFlip`, `RandomRotation(15°)`, and `ColorJitter(contrast=0.1)`

The validation set is only resized. Both then go through Keras'
`mobilenet_v2.preprocess_input`, which scales pixels from `[0, 255]` to
`[-1, 1]`, the range MobileNetV2 was trained on.

Keras keeps **channels-last** (`224, 224, 3`) with the PyTorch backend, so
`to_model_input` returns channels-last arrays rather than PyTorch's usual
`(3, 224, 224)` tensors. Doing this in the data pipeline keeps the saved
`.h5` model free of custom layers.

In [ ]:
def to_model_input(img):
    """ PIL image -> channels-last float array scaled to [-1, 1] """
    x = np.asarray(img, dtype="float32")  # (224, 224, 3) in [0, 255]
    return keras.applications.mobilenet_v2.preprocess_input(x)


train_augment = v2.Compose([
    v2.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    v2.RandomHorizontalFlip(),
    v2.RandomRotation(15),
    v2.ColorJitter(contrast=0.1),
])
val_resize = v2.Resize(IMG_SIZE)

train_transform = v2.Compose([train_augment, to_model_input])
val_transform = v2.Compose([val_resize, to_model_input])

# Same split indices, two views of the folder with different transforms
train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_transform),
                  train_idx)
val_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=val_transform),
                val_idx)

pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=pin)

x_batch, y_batch = next(iter(train_loader))
print("Batch:", tuple(x_batch.shape), x_batch.dtype,
      "| range: [{:.2f}, {:.2f}]".format(x_batch.min(), x_batch.max()))

> ### 🔍 Curiosity corner — what does augmentation do?
> Three training images: the plain resized original on the left, then
> random augmented versions. Every epoch the model sees a new random
> version, so it can't just memorise the pictures.

In [ ]:
N_VERSIONS = 5
rng = np.random.default_rng(1)
picks = rng.choice(train_idx, size=3, replace=False)
fig, axes = plt.subplots(3, N_VERSIONS + 1, figsize=(15, 8))
for row, i in zip(axes, picks):
    img, label = full_ds[int(i)]
    row[0].imshow(val_resize(img))
    row[0].set_title(f"original\n{class_names[label]}", fontsize=9)
    for k, ax in enumerate(row[1:], start=1):
        ax.imshow(train_augment(img))
        ax.set_title(f"augmented #{k}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Random crop + flip + rotation + contrast")
plt.tight_layout()
plt.show()

## 3. Build the model (frozen base)

MobileNetV2 with ImageNet weights and `include_top=False` (no 1000-class
ImageNet head). The base is frozen and called with `training=False`, so its
BatchNorm layers stay in inference mode. On top: global average pooling →
dropout → a new softmax layer with one output per Caltech-101 class.

This code is identical to the TensorFlow version: Keras 3 model code is
backend-independent.

In [ ]:
def build_model(num_classes):
    """ builds a frozen MobileNetV2 base with a new softmax head """
    base_model = keras.applications.MobileNetV2(
        weights=WEIGHTS, input_shape=IMG_SIZE + (3,), include_top=False)
    base_model.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base_model(inputs, training=False)  # keep BatchNorm in inference
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs), base_model


model, base_model = build_model(num_classes)
model.summary()

## 4. Phase 1 — train the classification head only

Only the new head is trainable. `EarlyStopping` restores the best weights
and `ReduceLROnPlateau` lowers the learning rate when validation loss stalls.
Keras 3 accepts PyTorch `DataLoader`s directly in `fit()`.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2),
]

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history_head = model.fit(train_loader, validation_data=val_loader,
                         epochs=EPOCHS_HEAD, callbacks=callbacks)

> ### 🔍 Curiosity corner — what does the new head predict?
> Validation images (never trained on) with the head's top guess and its
> confidence. Green = correct, red = wrong. The same helper is reused after
> fine-tuning, so you can compare.

In [ ]:
def show_predictions(model, n=12, seed=2):
    """ plots validation images with predicted vs true class """
    picks = np.random.default_rng(seed).choice(val_idx, size=n,
                                               replace=False)
    images = [val_resize(full_ds[int(i)][0]) for i in picks]
    labels = [full_ds.targets[int(i)] for i in picks]
    batch = np.stack([to_model_input(img) for img in images])
    probs = model.predict(batch, verbose=0)

    fig, axes = plt.subplots(3, n // 3, figsize=(14, 10))
    for ax, img, true, p in zip(axes.flat, images, labels, probs):
        pred = int(np.argmax(p))
        ax.imshow(img)
        ax.set_title(f"pred: {class_names[pred]} ({p[pred]:.0%})\n"
                     f"true: {class_names[true]}", fontsize=9,
                     color="green" if pred == true else "red")
        ax.axis("off")
    correct = sum(int(np.argmax(p)) == t for p, t in zip(probs, labels))
    plt.suptitle(f"{correct}/{n} correct")
    plt.tight_layout()
    plt.show()


show_predictions(model)

## 5. Phase 2 — fine-tune the top layers of the base model

The last `UNFREEZE_LAYERS` layers of MobileNetV2 are unfrozen, except
BatchNormalization layers: their running statistics were learned on
ImageNet and updating them on a small dataset tends to hurt accuracy.
The model is recompiled (required for the change to take effect) with a
100x smaller learning rate so the pretrained weights are only nudged.

In [ ]:
def unfreeze_top(base_model, n_layers):
    """ unfreezes the last n_layers of base_model, BatchNorm stays frozen """
    base_model.trainable = True
    split = max(len(base_model.layers) - n_layers, 0)
    for i, layer in enumerate(base_model.layers):
        is_bn = isinstance(layer, keras.layers.BatchNormalization)
        layer.trainable = i >= split and not is_bn


unfreeze_top(base_model, UNFREEZE_LAYERS)
print("Trainable layers in base:",
      sum(layer.trainable for layer in base_model.layers))

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history_finetune = model.fit(train_loader, validation_data=val_loader,
                             epochs=EPOCHS_FINETUNE, callbacks=callbacks)

> ### 🔍 Curiosity corner — predictions after fine-tuning
> Same 12 validation images as before. Did any red ones turn green?

In [ ]:
show_predictions(model)

## 6. Evaluate and save

Because `EarlyStopping` uses `restore_best_weights=True`, the model in memory
holds its best weights, so a single save at the end is enough.

In [ ]:
_, val_acc = model.evaluate(val_loader)
print("Validation accuracy: {:.4f}".format(val_acc))
model.save(MODEL_PATH)
print("Model saved as", MODEL_PATH)

## 7. The whole pipeline as `train_transfer_model()`

The task asks for a function `train_transfer_model()`. This cell wraps
sections 1–6 into that function, reusing the transforms from section 2 and
`build_model()` / `unfreeze_top()` defined above.

Running this cell only **defines** the function. Uncomment the last line to
train end to end in one call. That repeats the training already done in
sections 1–6, so it isn't needed if you ran those.

In [ ]:
def load_datasets():
    """ loads Caltech-101 as augmented train / plain validation loaders """
    folder = datasets.ImageFolder(DATA_DIR)
    num_classes = len(folder.classes)  # 101 objects + background
    perm = torch.randperm(len(folder),
                          generator=torch.Generator().manual_seed(42))
    n_val = int(0.2 * len(folder))
    val_idx, train_idx = perm[:n_val].tolist(), perm[n_val:].tolist()

    train_ds = Subset(
        datasets.ImageFolder(DATA_DIR, transform=train_transform), train_idx)
    val_ds = Subset(
        datasets.ImageFolder(DATA_DIR, transform=val_transform), val_idx)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin)
    return train_loader, val_loader, num_classes


def train_transfer_model():
    """ trains in two phases (head, then fine-tuning) and saves the model """
    train_loader, val_loader, num_classes = load_datasets()
    model, base_model = build_model(num_classes)
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2),
    ]

    # Phase 1: train the classification head only
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(train_loader, validation_data=val_loader, epochs=EPOCHS_HEAD,
              callbacks=callbacks)

    # Phase 2: fine-tune the top layers with a small learning rate
    unfreeze_top(base_model, UNFREEZE_LAYERS)
    model.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(train_loader, validation_data=val_loader,
              epochs=EPOCHS_FINETUNE, callbacks=callbacks)

    _, val_acc = model.evaluate(val_loader)
    print("Validation accuracy: {:.4f}".format(val_acc))
    model.save(MODEL_PATH)
    return model


# model = train_transfer_model()

### Verify the saved model

Reloads the `.h5` file to check it was written correctly.

In [ ]:
if os.path.exists(MODEL_PATH):
    loaded = keras.models.load_model(MODEL_PATH)
    print(f"'{MODEL_PATH}' saved and reloaded, output shape:",
          loaded.output_shape)
else:
    print(f"Error: the trained model '{MODEL_PATH}' was not found.")

!ls -F